# 1. 단어 사전 기반 매핑 복원

## 1) Import

In [ ]:
!pip install triton
!pip install transformers==4.41.2
!pip install accelerate==0.28.0 ### 0.31.0에서 LoRa때문에 버전 내림
!pip install -U bitsandbytes
!pip install peft==0.10.0

In [2]:
import pandas as pd
from tqdm import tqdm
import os
from google.colab import drive

## 2) Data Load

In [ ]:
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/data"
data_path = os.path.join(base_path, 'miniproj/')

In [4]:
train = pd.read_csv(data_path+'train.csv', encoding = 'utf-8-sig')
test = pd.read_csv(data_path+'test.csv', encoding = 'utf-8-sig')

In [ ]:
#train = pd.read_csv('./train.csv', encoding = 'utf-8-sig')
#test = pd.read_csv('./test.csv', encoding = 'utf-8-sig')

## 3) 단어 사전 생성

In [5]:
match_dict = {}

for input_text, output_text in zip(train['input'], train['output']):
    input_words = input_text.split()
    output_words = output_text.split()
    for iw, ow in zip(input_words, output_words):
        match_dict[iw] = ow

## 4) 변환 적용

In [6]:
def replace_words(input_text, match_dict):
    words = input_text.split()
    replaced_words = [match_dict.get(word, word) for word in words]
    return " ".join(replaced_words)

In [7]:
converted_reviews = test['input'].apply(lambda x: replace_words(x, match_dict)).tolist()

## 5) Submission

In [8]:
submission = pd.read_csv(data_path+'sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
#submission = pd.read_csv('./sample_submission.csv', encoding = 'utf-8-sig')

In [9]:
submission['output'] = converted_reviews

In [10]:
submission.to_csv(data_path+'baseline_submission.csv', index = False, encoding = 'utf-8-sig')

In [ ]:
#submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')

# 2. LLM 활용 (Gemma)

## 1) Import

In [11]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from accelerate import Accelerator
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset
from datetime import datetime

## 2) Data Load

In [12]:
train = pd.read_csv(data_path+'train.csv', encoding = 'utf-8-sig')
test = pd.read_csv(data_path+'test.csv', encoding = 'utf-8-sig')

In [ ]:
#train = pd.read_csv('./train.csv', encoding = 'utf-8-sig')
#test = pd.read_csv('./test.csv', encoding = 'utf-8-sig')

In [13]:
samples = []

for i in range(10):
    sample = f"input : {train['input'][i]} \n output : {train['output'][i]}"
    samples.append(sample)

## 3) Model Load

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type= 'nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
#model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
#model_id = 'beomi/gemma-ko-7b'
model_id = 'beomi/gemma-ko-2b'
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config = bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

3.5) LoRA  

In [15]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [16]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
print_trainable_parameters(model)

In [ ]:
print(model)

In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

In [18]:
def formatting_func(example):
    prompt = f"""
    You are a helpful assistant specializing in restoring obfuscated Korean reviews.
    Your task is to transform the given obfuscated Korean review into a clear, correct,
    and natural-sounding Korean review that reflects its original meaning.
    Below are examples of obfuscated Korean reviews and their restored forms:\n\n
    Example, {samples}
    Spacing and word length in the output must be restored to the same as in the input.
    Do not provide any description. Print only in Korean.\n\n
    input: {example['input']}\noutput: {example['output']}
    """
    return prompt

In [19]:
def generate_and_tokenize_prompt(prompt):
    return tokenizer(formatting_func(prompt), padding=False, truncation=False)


In [ ]:
dataset = load_dataset("csv", data_files=data_path+'train.csv')
full_dataset = dataset["train"]

split_dataset = full_dataset.train_test_split(test_size=0.2, seed=42)

split_dataset = {
    "train": split_dataset["train"],
    "eval": split_dataset["test"]
}

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["eval"]

In [ ]:
print(train_dataset)

In [ ]:
print(eval_dataset)

In [ ]:
tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = eval_dataset.map(generate_and_tokenize_prompt)

In [ ]:
print(tokenized_train_dataset)

In [ ]:
#model.gradient_checkpointing_disable()

In [44]:
torch.cuda.empty_cache()
#torch.cuda.reset_peak_memory_stats()

In [ ]:
project = "lora"
run_name = model_id + "-" + project
output_dir = "./" + run_name

training_args = TrainingArguments(
    output_dir="./lora_test",
    per_device_train_batch_size=1,
    #per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    #num_train_epochs = 1,
    max_steps = 500,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=10,
    report_to="none",
    save_strategy="no",
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator
)

model.config.use_cache = False
trainer.train()

print(f"✅ 학습 완료")

lora_output_dir = "./" + run_name + "_lora"
model.save_pretrained(lora_output_dir)
print(f"✅ LoRA weights 저장완료 ({lora_output_dir})")

In [ ]:
'''
from peft import PeftModel

model = PeftModel.from_pretrained(
    model_id,               # 원본 모델
    lora_output_dir,         # 학습된 LoRA weight 경로
    device_map="auto",   # GPU/CPU 자동 배치
    merge_weights=True   # LoRA merge
)
'''

In [ ]:
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

restored_reviews = []

for index, row in tqdm(test.head(10).iterrows(), desc="진행도"):
#for index, row in tqdm(test.iterrows(), desc="진행도"):
    query = row['input']

    messages = [
        {
            "role": "system",
            "content": f"""
            Transform the given obfuscated Korean review into a clear, correct, and natural-sounding Korean review that reflects its original meaning. Print only in Korean.
            Example, {samples}
            Spacing and word length in the output must be restored to the same as in the input.
            Do not provide any description. Print only in Korean.\n\n
            """
        },
        {
            "role": "user",
            "content": f"input : {query}, output : "
        },
    ]

    prompt = "\n".join([m["content"] for m in messages]).strip()


    outputs = pipe(
        prompt,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        max_new_tokens=len(query),
        eos_token_id=pipe.tokenizer.eos_token_id
    )

    generated_text = outputs[0]['generated_text']
    result = generated_text[len(prompt):].strip()


    restored_reviews.append(result)

In [ ]:
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(restored_reviews))
pd.reset_option('all')

## 5) Submission

In [ ]:
submission = pd.read_csv('./sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
submission['output'] = restored_reviews

In [ ]:
submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')